In [1]:
from langchain.chat_models import init_chat_model
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage

In [2]:
llm = init_chat_model("gemini-2.5-flash", model_provider="google_genai")

In [3]:
# Tools 
@tool()
def add(a: int, b: int) -> int:
    """Adds a and b."""
    return a + b

In [4]:
tools = [add]
llm_with_tools = llm.bind_tools(tools)

query = "What is 11 + 49?"
# query = "What is the capital of Australia?"

messages = [SystemMessage('You can use tools when necessary, otherwise answer the question using your own knowledge.'),
            HumanMessage(query)]

In [5]:
ai_msg = llm_with_tools.invoke(messages)
print(ai_msg)

content='' additional_kwargs={'function_call': {'name': 'add', 'arguments': '{"a": 11, "b": 49}'}, '__gemini_function_call_thought_signatures__': {'f0445df0-ba5e-420c-ab6e-d9ba48e723d6': 'CuIBAWkUfRORi5BS49YSo/DU6UQTReQEDHmLYSqqMbHDedTkH51c5a0oHwyHcEMtoSLKxTD7H5aRgzp3LNzPRSP3EXiIeVuNceIBgnU19fTlu/5sNRa0q+oNkfFEWEnf8BiGBxQ+PrVE3TsF9KxFnrL/vFXY3jIU9X2d99EeCt7e/3hzrV3sX3N4cORaw8GIHPyig/cs/IZpEuYBVGDHsq5ukUKxM2v2qXxMU2tNG52e0VSKQUDhRGjJdM6npHoG/5r8cBMn9xi77esm0ze/Ex+2OCmAOeiuP5gF2jZ/m7jENirk9QoiAWkUfROn0Ng16ES+xB2sY33SifowpqPRBWVxYwNXDii1KwooAWkUfRNqSfsfiN2hqcEmskN4kzQbyJx5oVFmpiY5eM+PwKjTvePLzAojAWkUfRNkxKWEnws4ZyrDF648F+9DXZLizsMlLkoL8xOhBhQ='}} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--01a10fd0-f504-7c03-bb24-e9922a63ff6a-0' tool_calls=[{'name': 'add', 'args': {'a': 11, 'b': 49}, 'id': 'f0445df0-ba5e-420c-ab6e-d9ba48e723d6', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={

In [6]:
# Add AI message to messages
messages.append(ai_msg)
# If tool call is present then call the tool and pass response to llm
if len(ai_msg.tool_calls) > 0:
    tool_call = ai_msg.tool_calls[-1]
    selected_tool = eval(tool_call['name'])  # Get function name to call

    # call the tool with required arguments provided by LLM in tool_call
    tool_msg = selected_tool.invoke(tool_call)
    messages.append(tool_msg)  # ToolMessage

    # Provide answer from tool back to LLM
    ai_message = llm_with_tools.invoke(messages)
    messages.append(ai_message)

In [7]:
for message in messages:
    message.pretty_print()

================================ System Message ================================

You can use tools when necessary, otherwise answer the question using your own knowledge.
================================ Human Message =================================

What is 11 + 49?
================================== Ai Message ==================================
Tool Calls:
  add (f0445df0-ba5e-420c-ab6e-d9ba48e723d6)
 Call ID: f0445df0-ba5e-420c-ab6e-d9ba48e723d6
  Args:
    a: 11
    b: 49
================================= Tool Message =================================
Name: add

60
================================== Ai Message ==================================

11 + 49 = 60.
